# English (Svarah) FastSpeech 2: TRAINING notebook (**GPU**)

**Use this only after the English alignment and features are done** (`preprocessed_Svarah.tar` on Drive). Cell 1 stops with a message if it is not there.

**Set the runtime to GPU first** (Runtime > Change runtime type > GPU).

| Cell | What it does | Time |
|---|---|---|
| 1 | Rebuilds the repo, config and features from Drive | a few minutes |
| 2 | Checks that a GPU is attached | seconds |
| 3 | **Smoke test**: 200 steps. Paste its output to Claude before cell 4 | a few minutes |
| 4 | **Real training**. Resumes automatically. Stop it any time | hours |
| 5 | Loss plots + checkpoint list | seconds |

If the runtime resets, run cells 1, 2 and 4 again: training resumes from the newest checkpoint on Drive.

## Cell 1: setup (restores everything from Drive)

In [1]:
import os
os.chdir("/content")                       # never start a cell inside a folder that may be deleted
LANG = "Svarah"                            # English
D = "/content/drive/MyDrive/tts_project"
from google.colab import drive
drive.mount('/content/drive')
import glob, re, shutil, subprocess
!pip install -q pyworld pypinyin unidecode inflect librosa pyyaml tgt tensorboard matplotlib scikit-learn

# the English features must exist on Drive before this notebook can run
tarp = f"{D}/preprocessed_Svarah.tar"
if not os.path.exists(tarp):
    raise SystemExit("preprocessed_Svarah.tar is not on Drive yet: the English alignment/feature step has not finished. Do that first (CPU).")

# a real repo has a text/ folder; a stub or half-deleted folder does not
if not os.path.isdir("/content/FastSpeech2/text"):
    shutil.rmtree("/content/FastSpeech2", ignore_errors=True)
    !git clone -q https://github.com/ming024/FastSpeech2.git /content/FastSpeech2
    !cd /content/FastSpeech2 && git apply {D}/repo_patches.diff && echo "repo patches applied"
    !cp -r {D}/config_Svarah /content/FastSpeech2/config/Svarah
    !cd /content/FastSpeech2/hifigan && unzip -o -q generator_universal.pth.tar.zip
P = "/content/FastSpeech2/preprocessed_data/Svarah"
if not os.path.exists(f"{P}/train.txt"):
    !mkdir -p /content/FastSpeech2/preprocessed_data
    !tar -xf {tarp} -C /content/FastSpeech2/preprocessed_data
os.chdir("/content/FastSpeech2")
for n in ["train.txt", "val.txt", "speakers.json", "stats.json"]:
    print(n, "OK" if os.path.exists(f"{P}/{n}") else "MISSING")
print("train lines:", sum(1 for _ in open(f"{P}/train.txt", encoding="utf-8")),
      "| val lines:", sum(1 for _ in open(f"{P}/val.txt", encoding="utf-8")))
print("vocoder weights:", os.path.exists("hifigan/generator_universal.pth.tar"))
!grep -nE "multi_speaker|max_seq_len|total_step|save_step|val_step|ckpt_path" config/Svarah/model.yaml config/Svarah/train.yaml

Mounted at /content/drive
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
repo patches applied
train.txt OK
val.txt OK
speakers.json OK
stats.json OK
train lines: 3323 | val lines: 657
vocoder weights: True
config/Svarah/model.yaml:22:multi_speaker: true
config/Svarah/model.yaml:23:max_seq_len: 1500
config/Svarah/train.yaml:2:  ckpt_path: /content/drive/MyDrive/tts_project/output/ckpt/Svarah
config/Svarah/train.yaml:21:  total_step: 15000
config/Svarah/train.yaml:24:  val_step: 1500
config/Svarah/train.yaml:25:  save_step: 3000


## Cell 2: GPU check

In [2]:
import torch
print("GPU available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print(torch.cuda.get_device_name(0))
else:
    raise SystemExit("No GPU attached. Runtime > Change runtime type > GPU, then re-run cell 1.")

GPU available: True
Tesla T4


## Cell 3: smoke test (200 steps). Paste the output before running cell 4

In [3]:
# Smoke test: 200 steps incl. validation, checkpoint and audio synthesis. Writes to LOCAL disk only.
import yaml
os.chdir("/content/FastSpeech2")
c = yaml.safe_load(open("config/Svarah/train.yaml"))
c["path"].update(ckpt_path="/content/smoke/ckpt", log_path="/content/smoke/log", result_path="/content/smoke/result")
c["step"].update(total_step=200, log_step=20, synth_step=100, val_step=100, save_step=100)
yaml.safe_dump(c, open("/content/smoke_train.yaml", "w"), sort_keys=False)
!cd /content/FastSpeech2 && python train.py -p config/Svarah/preprocess.yaml -m config/Svarah/model.yaml -t /content/smoke_train.yaml > /content/smoke.log 2>&1; echo "exit code: $?"
!tr '\r' '\n' < /content/smoke.log | grep -a -E "Number of|Total Loss|Validation|Error|Traceback|error" | tail -25
print("---- last lines ----")
!tr '\r' '\n' < /content/smoke.log | tail -4
!ls -la /content/smoke/ckpt

exit code: 0
Number of FastSpeech2 Parameters: 35430721
Epoch 1:  94%|█████████▍| 49/52 [02:21<00:06,  2.29s/it]Step 20/200, Total Loss: 16.2698, Mel Loss: 4.3937, Mel PostNet Loss: 4.3738, Pitch Loss: 1.9104, Energy Loss: 2.3592, Duration Loss: 3.2328
Step 40/200, Total Loss: 12.3342, Mel Loss: 3.9781, Mel PostNet Loss: 3.5850, Pitch Loss: 1.3651, Energy Loss: 1.7513, Duration Loss: 1.6547
Step 60/200, Total Loss: 11.5975, Mel Loss: 3.6665, Mel PostNet Loss: 3.0459, Pitch Loss: 2.1856, Energy Loss: 1.6299, Duration Loss: 1.0696
Step 80/200, Total Loss: 9.7002, Mel Loss: 3.6641, Mel PostNet Loss: 2.8795, Pitch Loss: 0.8509, Energy Loss: 1.4034, Duration Loss: 0.9022
Step 100/200, Total Loss: 10.0663, Mel Loss: 3.6983, Mel PostNet Loss: 2.9106, Pitch Loss: 0.9212, Energy Loss: 1.6623, Duration Loss: 0.8739
Validation Step 100, Total Loss: 8.3355, Mel Loss: 3.6229, Mel PostNet Loss: 2.5450, Pitch Loss: 0.6314, Energy Loss: 0.8919, Duration Loss: 0.6443
Step 120/200, Total Loss: 9.3437, M

## Cell 4: real training (GPU)

In [4]:
# REAL TRAINING. Resumes from the newest checkpoint on Drive. Stop the cell any time; the newest checkpoint is kept.
os.chdir("/content/FastSpeech2")
steps = [int(re.findall(r"(\d+)\.pth\.tar$", p)[0]) for p in glob.glob(f"{D}/output/ckpt/{LANG}/*.pth.tar")]
last = max(steps) if steps else 0
print("resuming from step", last)
!cd /content/FastSpeech2 && python train.py --restore_step {last} -p config/Svarah/preprocess.yaml -m config/Svarah/model.yaml -t config/Svarah/train.yaml

resuming from step 0
I0000 00:00:1794054369.779651    3443 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
I0000 00:00:1794054372.066932    3443 cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.
Prepare training ...
Number of FastSpeech2 Parameters: 35430721
/usr/local/lib/python3.13/dist-packages/torch/nn/utils/weight_norm.py:145: FutureWarning: `torch.nn.utils.weight_norm` is deprecated in favor of `torch.nn.utils.parametrizations.weight_norm`.
  WeightNorm.apply(module, name, dim)
Removing weight norm...
Training:   0% 0/15000 [00:00<?, ?it/s]
Training:   0% 5/15000 [00:03<2:09:01,  1.94it/s]
Training:   0% 9/15000 [00:05<1:56:11,  2.15it/s]
Training:   0% 13/15000 [00:07<1:45:41,  2.36it/s]
Training:   0% 17/15000 [00:09<1:50:56,  2.

## Cell 5: loss plots and checkpoints

In [5]:
# Progress + loss curves from the log files on Drive (safe to run any time)
import re, os, matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np
OUT = f"{D}/Results/English"; os.makedirs(OUT, exist_ok=True)
NAMES = ["Total", "Mel", "Mel PostNet", "Pitch", "Energy", "Duration"]

def parse(path):
    rows = []
    if not os.path.exists(path): return np.array(rows)
    for line in open(path):
        m = re.search(r"Step\s+(\d+)", line)
        vals = [re.search(rf"(?<![A-Za-z]){n} Loss:\s*([0-9.eE+-]+)", line) for n in NAMES]
        if m and all(vals): rows.append([int(m.group(1))] + [float(v.group(1)) for v in vals])
    return np.array(rows)

def plot(rows, title, fname, smooth=0):
    if len(rows) == 0:
        print("no parsable lines for", title); return
    fig, ax = plt.subplots(1, 2, figsize=(13, 4.5))
    for i, n in enumerate(NAMES):
        y = rows[:, 1 + i]
        if smooth and len(y) > smooth: y = np.convolve(y, np.ones(smooth) / smooth, mode="valid"); x = rows[smooth - 1:, 0]
        else: x = rows[:, 0]
        (ax[0] if n == "Total" else ax[1]).plot(x, y, label=n)
    ax[0].set_title(title + " (total)"); ax[1].set_title(title + " (components)")
    for a in ax: a.set_xlabel("step"); a.set_ylabel("loss"); a.legend(); a.grid(alpha=.3)
    plt.tight_layout(); plt.savefig(f"{OUT}/{fname}", dpi=150); plt.close()
    print("saved", f"{OUT}/{fname}", "| points:", len(rows), "| last step:", int(rows[-1, 0]))

LOG = f"{D}/output/log/Svarah"
tr = parse(f"{LOG}/train/log.txt"); va = parse(f"{LOG}/val/log.txt")
plot(tr, "English training loss", "training_loss.png", smooth=5)
plot(va, "English validation loss", "validation_loss.png")
if len(va): print("best validation total loss: %.4f at step %d" % (va[:, 1].min(), va[va[:, 1].argmin(), 0]))
print("checkpoints on Drive:", sorted(int(re.findall(r"(\d+)\.pth\.tar$", p)[0]) for p in glob.glob(f"{D}/output/ckpt/Svarah/*.pth.tar")))

saved /content/drive/MyDrive/tts_project/Results/English/training_loss.png | points: 150 | last step: 15000
saved /content/drive/MyDrive/tts_project/Results/English/validation_loss.png | points: 10 | last step: 15000
best validation total loss: 2.6394 at step 9000
checkpoints on Drive: [3000, 6000, 9000, 12000, 15000]
